# 02 Model A: plain age regression on frozen CLIP ViT-L/14 embeddings

Needs notebook 01 done (UTKFace embeddings, Phase 1 cross-check PASS, folds). Writes per-image OOF predictions to Drive only and an aggregate report JSON. Charts only; no images are opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions from notebook 01

In [ ]:
chk = json.loads((DRIVE_ROOT / 'results/embeddings/check_vs_phase1.json').read_text())
assert chk['pass'], 'Phase 1 cross-check did not pass: stop'
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
assert (DRIVE_ROOT / 'results/folds/utkface_folds_seed0.json').exists(), 'folds missing'
print('cross-check max diff', chk['max_abs_diff'], '| decisions agree', chk['decision_agreement'])

## 2. Train model A (5 folds x 8 grid settings; a few minutes on T4)

In [ ]:
!python scripts/train_model.py --model A
d = save_manifest('models/A.json')
r = d['report']
print('A self-check:', d['a_self_check'])
print('worst-group TPR', r['worst_group_tpr_strat'], '\nadult FPR B-W', r['fpr_diff_B_minus_W'])

## 3. Aggregate outputs for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step6_outputs.zip'
files = [DRIVE_ROOT / 'results/models/A.json']
with zipfile.ZipFile(out, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(DRIVE_ROOT))
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))